In [ ]:
import numpy as np
import os
import pandas as pd 
from sklearn.metrics import confusion_matrix
#import seaborn as sn; sn.set(font_scale=1.4)
from sklearn.utils import shuffle           
import matplotlib.pyplot as plt             
import cv2                                 
import tensorflow as tf                
from tqdm import tqdm
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, Flatten, Embedding
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.optimizers import SGD, Adam
import numpy as np
from keras.datasets import mnist
from keras.utils import np_utils,to_categorical
from keras.models import Sequential
from keras.optimizers import Adam
import numpy as np 
import pandas as pd 
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Activation, Conv2D, MaxPooling2D,BatchNormalization,GlobalAveragePooling2D
from tqdm import tqdm
from sklearn.utils import shuffle
from tensorflow.keras.utils import load_img,img_to_array
import random
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
#%%
#import image data from google drive
!gdown --id '1qCHfycy91EyUFzilBvxu8hYVYp0l6jCh' --output temp/Midterm_dataset.zip

from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/My Drive/MURA')

'定義子資料夾名稱&對應的數字，檔名不能有中文'
class_names = ['ELBOW_negative','SHOULDER_negative','WRIST_negative']
class_names_label = {class_name:i for i, class_name in enumerate(class_names)}
nb_classes = len(class_names)

IMAGE_SIZE = (224, 224)

def load_data():
    datasets = ['train', 'test', 'valid']#資料夾
    output = []
    
    # Iterate through training and test sets
    for dataset in datasets:
        
        images = []
        labels = []
        
        print("Loading {}".format(dataset))
        
        # Iterate through each folder corresponding to a category
        for folder in os.listdir(dataset):
            label = class_names_label[folder]
            
            # Iterate through each image in our folder
            for file in tqdm(os.listdir(os.path.join(dataset, folder))):
                
                # Get the path name of the image
                img_path = os.path.join(os.path.join(dataset, folder), file)
                
                # Open and resize the img
                image = cv2.imread(img_path)
                #print(file,image)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                #cv讀照片，顏色莫認為BGR，需轉為RGB，錯誤表示黑白或已轉
                image = cv2.resize(image, IMAGE_SIZE)
                
                # Append the image and its corresponding label to the output
                images.append(image)
                labels.append(label)
                
        images = np.array(images, dtype = 'float32')
        labels = np.array(labels, dtype = 'int32')   
        output.append((images, labels))
    return output


(x_train,y_train),(x_test,y_test),(x_valid, y_valid) = load_data()
x_train = x_train.reshape(x_train.shape[0],224,224,1).astype('float32')
x_test = x_test.reshape(x_test.shape[0],224,224,1).astype('float32')
x_valid = x_valid.reshape(x_valid.shape[0],224,224,1).astype('float32')

print("x_train shape: ",x_train.shape)
print("x_test shape: ",x_test.shape)

x_train = x_train/255
x_test = x_test/255
x_valid = x_valid/255

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
y_valid = to_categorical(y_valid)

(x_train,y_train) = shuffle(x_train,y_train)
(x_test,y_test) = shuffle(x_test,y_test)
(x_valid, y_valid) = shuffle(x_valid, y_valid)

#model = VGG16()

model = Sequential()
# Adds a densely-connected layer with 64 units to the model:
model.add(Conv2D(64,(3,3), activation = 'relu', input_shape = (224,224,1)))
model.add(Conv2D(64,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(64,(3,3), activation = 'relu', padding = 'same'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Conv2D(128,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(128,(3,3), activation = 'relu', padding = 'same'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Conv2D(256,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(256,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(256,(3,3), activation = 'relu', padding = 'same'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(Conv2D(512,(3,3), activation = 'relu', padding = 'same'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Flatten())
model.add(Dense(4096, activation='relu'))
model.add(Dense(4096, activation='relu'))
model.add(BatchNormalization()) 
# Add a softmax layer with 10 output units:
model.add(Dense(3, activation='softmax'))

model.summary()
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
history = model.fit(x_train, y_train ,shuffle = True,validation_data = (x_valid, y_valid), epochs=50, batch_size=32,verbose = 1)



# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

def plot_images_labels(images,labels,prediction,idx,num=15):
    fig=plt.gcf()
    fig.set_size_inches(12,14)
    if num>25:
        num=25
    for i in range(0,num):
        ax=plt.subplot(5,5,1+i)
        ax.imshow(np.reshape(images[idx],(224,224,1)), cmap='binary')
        if np.argmax(labels,axis=1)[idx] == 0:
            title="label=elbow" 
            if len(prediction)>=0:
                if np.argmax(prediction,axis=1)[idx] == 0 :
                    title+=",predict=elbow"
                if np.argmax(prediction,axis=1)[idx] == 1 :
                    title+=",predict=shoulder"
                if np.argmax(prediction,axis=1)[idx] == 2 :
                    title+=",predict=wrist"
        if np.argmax(labels,axis=1)[idx] == 1:
            title="label=shoulder" 
            if len(prediction)>=0:
                if np.argmax(prediction,axis=1)[idx] == 0 :
                    title+=",predict=elbow"
                if np.argmax(prediction,axis=1)[idx] == 1 :
                    title+=",predict=shoulder"
                if np.argmax(prediction,axis=1)[idx] == 2 :
                    title+=",predict=wrist"
        if np.argmax(labels,axis=1)[idx] == 2:
            title="label=wrist" 
            if len(prediction)>=0:
                if np.argmax(prediction,axis=1)[idx] == 0 :
                    title+=",predict=elbow"
                if np.argmax(prediction,axis=1)[idx] == 1 :
                    title+=",predict=shoulder"
                if np.argmax(prediction,axis=1)[idx] == 2 :
                    title+=",predict=wrist"
        ax.set_title(title,fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        idx=idx+1
    plt.show()
    
#prediction=model.predict_classes(x_test)
prediction=model.predict(x_test) 
classes_x=np.argmax(prediction,axis=1)

plot_images_labels(x_test,y_test,prediction,idx=300)

score = model.evaluate(x_test,y_test)
'''
predicted_val = [int(round(p)) for p in prediction]
labelss = y_test[0:503][1]
df_1 = pd.DataFrame({"Label" : labelss ,"predict" : predicted_val})
df_2 = df_1[labelss != predicted_val]
print(df_2.head())
'''
y_test_bk = y_test.copy()
y_pred = np.argmax(prediction,axis=1)
tb = pd.crosstab(np.argmax(y_test_bk,axis=1).astype(int),y_pred.astype(int),rownames = ["label"], colnames = ["predict"])
print(tb)

# 新增區段

# 新增區段